# CC6: Wales electronic-component employment

Live data download and replication notebook. ONS Business Register and Employment Survey, open-access dataset NM_189_1, SIC 26110. Employee jobs at workplace, 2015–2024. This industry includes semiconductors and other electronic components; it is not a complete semiconductor workforce count.

The Python loop downloads six original JSON responses, preserves them unchanged and records request URLs, timestamps and SHA-256 hashes. Run all cells in a standard Python 3 Colab runtime. No API key, Drive mount, pip installation or credentials required.

Technical AI assistance: Codex helped implement data retrieval, cleaning, checks and visualisation code. Analytical interpretation remains Conrad’s.


In [1]:
from pathlib import Path
import json, urllib.request, urllib.parse, hashlib, datetime, time, zipfile
OUTPUT = Path("cc6-wales-output")
(OUTPUT / "raw").mkdir(parents=True, exist_ok=True)
(OUTPUT / "clean").mkdir(exist_ok=True)
AREAS = [
    ("newport", "W06000022", "Newport"),
    ("cardiff", "W06000015", "Cardiff"),
    ("monmouthshire", "W06000021", "Monmouthshire"),
    ("wrexham", "W06000006", "Wrexham"),
    ("flintshire", "W06000005", "Flintshire"),
    ("torfaen", "W06000020", "Torfaen"),
]
BASE = "https://www.nomisweb.co.uk/api/v01/dataset/NM_189_1.data.json"


## 1. Download six series

Nomis interprets the requested endpoints as the annual interval 2015–2024. Each response is checked below against the full expected year sequence.


In [2]:
receipts = []
for slug, code, name in AREAS:
    params = dict(geography=code, industry=134243838, employment_status=1,
                  measure=1, measures=20100, time="2015,2024")
    url = BASE + "?" + urllib.parse.urlencode(params, safe=",")
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "PP434 course data retrieval"})
            raw = urllib.request.urlopen(request, timeout=20).read()
            source = json.loads(raw)
            assert source.get("obs"), "No observations returned"
            assert str(source["header"].get("truncated", "false")).lower() == "false", "Truncated response"
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)
    path = OUTPUT / "raw" / (slug + ".json")
    path.write_bytes(raw)
    receipts.append(dict(area=name, code=code, slug=slug, url=url,
                         retrieved_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                         source_prepared=source["header"].get("prepared"),
                         sha256=hashlib.sha256(raw).hexdigest(), bytes=len(raw)))
    print(name + ": downloaded " + str(len(source["obs"])) + " observations as raw JSON")
assert len(receipts) == 6


Newport: downloaded 10 observations as raw JSON
Cardiff: downloaded 10 observations as raw JSON
Monmouthshire: downloaded 10 observations as raw JSON
Wrexham: downloaded 10 observations as raw JSON
Flintshire: downloaded 10 observations as raw JSON
Torfaen: downloaded 10 observations as raw JSON


## 2. Clean and validate

Preserve the original observations separately. Reject missing observations or unexpected geography/industry codes; retain years 2015–2024 and check uniqueness.


In [3]:
combined = []
for receipt in receipts:
    source = json.loads((OUTPUT / "raw" / (receipt["slug"] + ".json")).read_bytes())
    rows = []
    for o in source["obs"]:
        year = int(o["time"]["value"])
        if not 2015 <= year <= 2024:
            continue
        assert o["geography"]["geogcode"] == receipt["code"]
        assert int(o["industry"]["value"]) == 134243838
        assert o["obs_status"]["value"] == "A", "Missing/non-normal observation"
        rows.append(dict(year=year, area=receipt["area"], code=receipt["code"],
                         employees=float(o["obs_value"]["value"]), status=o["obs_status"]["description"]))
    rows.sort(key=lambda r: r["year"])
    assert [r["year"] for r in rows] == list(range(2015, 2025)), "Missing or duplicate years"
    baseline = rows[0]["employees"]
    for row in rows:
        row["index_2015"] = round(row["employees"] / baseline * 100, 2) if baseline else None
    (OUTPUT / "clean" / (receipt["slug"] + ".json")).write_text(json.dumps(rows, indent=2))
    combined.extend(rows)
assert len(combined) == 60
(OUTPUT / "timeseries.json").write_text(json.dumps(combined, indent=2))
(OUTPUT / "source-manifest.json").write_text(json.dumps(receipts, indent=2))
print("Validated: 6 areas × 10 annual observations = 60. No missing years or non-normal observations.")


Validated: 6 areas × 10 annual observations = 60. No missing years or non-normal observations.


## 3. Package replication files


In [4]:
archive = OUTPUT / "CC6_Wales_data.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUTPUT.rglob("*.json")):
        z.write(file, file.relative_to(OUTPUT))
print("Ready:", archive)
# In Colab, use the Files panel to download CC6_Wales_data.zip.
# Raw JSON belongs in data/wales-semiconductors/sources/ in the site repository.
# Clean area JSON and timeseries.json belong in data/wales-semiconductors/.


Ready: cc6-wales-output/CC6_Wales_data.zip
